# Round 11 — A Cross-Sectional Veto

Reproduces the round end to end. Every number in `LOG.md` section 78 comes from here.

**The formulation is `research/penny/FORMULATION.md`; this notebook implements it.
Where they disagree, this notebook is wrong.**

## The problem with rounds 1-10

Each estimated the unconditional mean of a screened set. Decompose the forward
return of stock $i$ entered on day $t$:

$$R_{i,t} \;=\; \mu_t \;+\; \lambda_{t,g(i)} \;+\; \eta_{i,t}$$

A screen firing on few days puts nearly all its variance in the **day** term
$\mu_t$. Measured in section 76: 286 trades on **48 days**, paired control
$-1.06\%$ with a CI **22 points wide**. "No effect" and "an effect too small to
see" were indistinguishable.

## The fix

Demean within the day cell:

$$\tilde R_{i,t} \;=\; R_{i,t} - \frac{1}{|c|}\sum_{j\in c} R_{j,t}$$

$\mu_t$ cancels **algebraically** - no control group, no matching. Cells are days
(Amendment 1: day-by-sector cells have median size 1-2). This takes the design
from 48 clusters to **1,834**.

In [1]:
import os, sys, warnings
# Data paths are repo-root relative, so anchor the working directory there.
# This makes the notebook runnable from anywhere, including nbconvert.
while not os.path.isdir("data/research_penny") and os.path.basename(os.getcwd()):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 160)

from research.penny.scripts.run_blocks import build_frame
from research.penny.scripts.run_composite import add_composite
from research.penny.strategy.cross_section import score_feature, MIN_CELL

H = 60                              # holding horizon, sessions
SPLIT = pd.Timestamp("2020-01-01")  # signs fitted before this, reported after
SEAL = pd.Timestamp("2024-01-01")   # holdout, read exactly once

## 1. The population

Amendment 1 fixes it: eligible symbol-days that fell at least 15% over five
sessions, price 0.50-10 dollars, ADV at least 250k, no corporate action, lookback
span at most 18 calendar days, in a day-cell of at least five names.

`build_frame` also computes the outcome - the plain h-session forward return from
the **next open**, net of the measured cost schedule. No barriers: exit
engineering is a separate question and mixing it in would confound signal quality
with it.

In [2]:
df = build_frame(H, 0.15)
print(f"candidates {len(df):,} | cells {df['date'].nunique():,} | "
      f"median cell {int(df.groupby('date').size().median())}")
print(f"raw mean R = {100*df['R'].mean():+.2f}%   <- this is mu_t, NOT the estimand")
df[["symbol", "date", "R", "Rt", "cost"]].head()

candidates 57,129 | cells 1,829 | median cell 17
raw mean R = +13.49%   <- this is mu_t, NOT the estimand


,symbol,date,R,Rt,cost
0,AA,2020-03-10,0.359342,-0.046657,0.0045
1,AA,2020-03-11,0.549991,-0.090101,0.0045
2,AA,2020-03-12,0.553668,0.059631,0.0045
3,AA,2020-03-13,0.749102,0.035166,0.0045
4,AA,2020-03-16,0.458983,-0.079262,0.0045


## 2. Block scores

`B0_random` is a **kill rule**: a pure random feature must score zero. If it does
not, the estimator is broken and nothing else means anything.

In [3]:
rows = []
for f in sorted(c for c in df.columns if c.startswith("B")):
    r = score_feature(df, f)
    if r:
        rows.append(dict(feature=f, cells=r.n_cells, IC=round(r.ic, 4),
                         lo=round(r.lo, 4), hi=round(r.hi, 4),
                         t=round(r.t_stat, 2), LS_net=round(100 * r.ls_net, 2)))
sc = pd.DataFrame(rows).sort_values("t")
display(sc)

ctrl = sc[sc.feature == "B0_random"].iloc[0]
print("control:", "PASSES" if ctrl.lo < 0 < ctrl.hi else "FAILS -- estimator broken")
print(f"cost floor (random LS net) = {ctrl.LS_net:+.2f}%"
      "  <- read every LS_net against THIS, not zero")

,feature,cells,IC,lo,hi,t,LS_net
10,B7_realised_vol,1829,-0.0968,-0.1113,-0.0830,-13.58,-2.04
9,B7_prior_spikes,1484,-0.0910,-0.1059,-0.0760,-11.71,-4.54
12,B8_dist_from_low,1829,-0.0684,-0.0824,-0.0544,-9.62,-1.39
13,B8_pct_of_range,1829,-0.0354,-0.0496,-0.0218,-4.94,-1.17
5,B2_idio_depth,1829,-0.0290,-0.0422,-0.0155,-4.26,-1.55
0,B0_crash_depth,1829,-0.0280,-0.0410,-0.0149,-4.13,-1.35
2,B1_overnight_share,1829,-0.0157,-0.0292,-0.0027,-2.26,-2.84
6,B2_sector_move,1828,-0.0114,-0.0253,0.0024,-1.60,-1.64
4,B1_worst_day_share,1829,-0.0067,-0.0199,0.0071,-0.94,-1.71
11,B8_consec_down,1827,0.0045,-0.0089,0.0185,0.65,-0.84


control: PASSES
cost floor (random LS net) = -0.72%  <- read every LS_net against THIS, not zero


Signs match the literature named in the formulation *before* the test: low
realised vol wins (Bali-Cakici-Whitelaw MAX effect), lottery names underperform,
short-horizon reversal (Jegadeesh, Lehmann), attention (Barber-Odean), liquidity
conditioning (Avramov-Chordia-Goyal).

## 3. The composite

Signs fitted on development **before 2020**, everything reported **from 2020**.

In [4]:
SIGNS = {"B7_realised_vol": -1, "B7_prior_spikes": -1, "B8_dist_from_low": -1,
         "B3_dv_shock": 1, "B3_log_adv": 1, "B1_range_expansion": 1,
         "B8_pct_of_range": -1}

te = add_composite(df[df.date >= SPLIT].copy(), SIGNS, "COMPOSITE")
te = add_composite(te, {k: -v for k, v in SIGNS.items()}, "ANTI")
te["RAND"] = np.random.default_rng(0).standard_normal(len(te))
print("composite IC:", round(score_feature(te, "COMPOSITE").ic, 4))

composite IC: 0.0999


## 4. The signal is monotone but **left-tailed**

The obvious construction - long the top decile - does not work. Reversing the
signs does. That asymmetry is the whole finding: the composite identifies
**losers**, not winners, so the tradable form is a **veto**.

In [5]:
def excess(sub, feat, keep, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for d, g in sub.groupby("date", sort=False):
        if len(g) < MIN_CELL:
            continue
        k = max(1, int(np.ceil(keep * len(g))))
        sel = g.nlargest(k, feat) if keep < 1 else g
        rows.append((float(sel.R.mean() - 0.5 * sel.cost.mean()),
                     float(g.R.mean() - 0.5 * g.cost.mean())))
    a = np.array(rows)
    d_ = a[:, 0] - a[:, 1]
    b = d_[rng.integers(0, len(a), size=(5000, len(a)))].mean(axis=1)
    return dict(cells=len(a), port=100 * a[:, 0].mean(), cell=100 * a[:, 1].mean(),
                excess=100 * d_.mean(), lo=100 * np.percentile(b, 2.5),
                hi=100 * np.percentile(b, 97.5))


out = []
for feat in ("COMPOSITE", "ANTI", "RAND"):
    for keep, lab in ((0.10, "long top decile"), (0.75, "drop worst 25%"),
                      (0.90, "drop worst 10%")):
        out.append(dict(feature=feat, form=lab,
                        **{k: round(v, 2) for k, v in excess(te, feat, keep).items()}))
pd.DataFrame(out)

,feature,form,cells,port,cell,excess,lo,hi
0,COMPOSITE,long top decile,973,5.22,6.4,-1.18,-3.20,1.01
1,COMPOSITE,drop worst 25%,973,7.47,6.4,1.07,0.56,1.55
2,COMPOSITE,drop worst 10%,973,7.07,6.4,0.67,0.44,0.90
3,ANTI,long top decile,973,0.77,6.4,-5.63,-7.99,-3.30
4,ANTI,drop worst 25%,973,6.64,6.4,0.24,-0.22,0.69
5,ANTI,drop worst 10%,973,6.54,6.4,0.14,-0.02,0.30
6,RAND,long top decile,973,4.98,6.4,-1.42,-3.70,0.96
7,RAND,drop worst 25%,973,6.24,6.4,-0.16,-0.67,0.32
8,RAND,drop worst 10%,973,6.38,6.4,-0.02,-0.24,0.19


## 5. The veto, and the holdout

The holdout is read **once**. Signs are never refitted.

In [6]:
from research.penny.scripts.veto_holdout import frame as hframe

ho = add_composite(hframe(H, 0.15, "holdout"), SIGNS, "COMPOSITE")
ho["RAND"] = np.random.default_rng(1).standard_normal(len(ho))

res = []
for name, sub in (("development >=2020", te), ("SEALED HOLDOUT 2024-26", ho)):
    for feat in ("COMPOSITE", "RAND"):
        e = excess(sub, feat, 0.75)
        res.append(dict(period=name, feature=feat,
                        **{k: round(v, 2) for k, v in e.items()}))
pd.DataFrame(res)

,period,feature,cells,port,cell,excess,lo,hi
0,development >=2020,COMPOSITE,973,7.47,6.40,1.07,0.56,1.55
1,development >=2020,RAND,973,6.24,6.40,-0.16,-0.67,0.32
2,SEALED HOLDOUT 2024-26,COMPOSITE,648,3.05,0.62,2.43,1.93,2.93
3,SEALED HOLDOUT 2024-26,RAND,648,0.67,0.62,0.05,-0.38,0.47


In [7]:
ho["y"] = pd.DatetimeIndex(ho.date).year
for y in sorted(ho.y.unique()):
    e = excess(ho[ho.y == y], "COMPOSITE", 0.75)
    print(f"{y}  cells={e['cells']:>4}  base {e['cell']:+6.2f}%  "
          f"veto {e['port']:+6.2f}%  excess {e['excess']:+5.2f}%  "
          f"CI [{e['lo']:+.2f}%, {e['hi']:+.2f}%]")

def annualise(per_cell_returns):
    """Positions overlap: 1/H of capital enters every session, so the account's
    daily return is the average DAILY-EQUIVALENT of the live holds. Compounding
    the mean 60-day return instead would apply a convex transform to a mean and
    overstate the result -- for the holdout that is +13.4% against +5.9%."""
    daily = (1 + per_cell_returns) ** (1 / H) - 1
    ann = (1 + daily.mean()) ** 252 - 1
    sharpe = daily.mean() / daily.std() * np.sqrt(252) if daily.std() > 0 else 0.0
    return 100 * ann, sharpe


for name, sub in (("dev", te), ("holdout", ho)):
    rows = []
    for d, g in sub.groupby("date", sort=False):
        if len(g) < MIN_CELL:
            continue
        k = max(1, int(np.ceil(0.75 * len(g))))
        sel = g.nlargest(k, "COMPOSITE")
        rows.append((float(g.R.mean() - 0.5 * g.cost.mean()),
                     float(sel.R.mean() - 0.5 * sel.cost.mean())))
    a = pd.DataFrame(rows, columns=["base", "veto"])
    for lab in ("base", "veto"):
        ann, sh = annualise(a[lab])
        print(f"  {name:>8} {lab:>5}: per-60d {100*a[lab].mean():+6.2f}%  ->  "
              f"annualised {ann:+6.2f}%  Sharpe {sh:5.2f}")

2024  cells= 251  base  -0.40%  veto  +2.18%  excess +2.58%  CI [+1.70%, +3.41%]


2025  cells= 249  base  +3.56%  veto  +6.19%  excess +2.63%  CI [+1.83%, +3.44%]


2026  cells= 148  base  -2.60%  veto  -0.77%  excess +1.83%  CI [+0.91%, +2.65%]


       dev  base: per-60d  +6.40%  ->  annualised +10.23%  Sharpe  1.35
       dev  veto: per-60d  +7.47%  ->  annualised +14.93%  Sharpe  1.92


   holdout  base: per-60d  +0.62%  ->  annualised  -4.12%  Sharpe -0.88
   holdout  veto: per-60d  +3.05%  ->  annualised  +5.94%  Sharpe  1.21


## 6. What did **not** work

Blocks B4 (dilution), B5 (filings) and B6 (short interest) - the only non-price
information in the project - add nothing. Round 2 claimed filings were the
largest lift ever measured here; round 4 showed that was mostly a fold artifact,
and this confirms it independently with a far better estimator.

In [8]:
from research.penny.scripts.blocks_fundamental import add_fundamental_blocks

dfx = add_fundamental_blocks(build_frame(H, 0.15))
rows = []
for f in sorted(c for c in dfx.columns if c.startswith(("B4_", "B5_", "B6_"))):
    r = score_feature(dfx, f)
    if r:
        rows.append(dict(feature=f, IC=round(r.ic, 4), t=round(r.t_stat, 2),
                         lo=round(r.lo, 4), hi=round(r.hi, 4)))
pd.DataFrame(rows).sort_values("t")

,feature,IC,t,lo,hi
2,B5_days_since_filing,-0.0133,-1.98,-0.0265,-0.0005
1,B5_days_since_deficiency,-0.0096,-1.41,-0.0229,0.0033
0,B4_shares_growth_1y,-0.0090,-1.09,-0.0245,0.0075
7,B6_short_ratio_20d,-0.0014,-0.19,-0.0159,0.0127
4,B5_n_shelf_180d,0.0008,0.13,-0.0120,0.0134
6,B6_short_ratio,0.0030,0.40,-0.0112,0.0179
5,B5_n_takedown_90d,0.0030,0.44,-0.0103,0.0162
3,B5_n_form4_90d,0.0057,0.85,-0.0071,0.0193


## Conclusion

A **veto** - drop the worst 25% of each day's crashed names by the composite -
converts a population that loses **-4.12% annualised** on the holdout into
**+5.94%, Sharpe 1.21**, with the excess positive and significant in all three
holdout years.

**Honest limits** (LOG section 78): the magnitude is modest; it is a veto, not a
selector; the base is negative so the veto carries the entire return; 2026 was
still -0.77% after vetoing; and execution at ~58 simultaneous microcap positions
is assumed rather than measured.